In [1]:
# =============================================================================
# DINO (ViT-S/16) — FULL PIPELINE: PRE-TRAINING + EVALUATION
# Pre-train on PatternNet → Evaluate on EuroSAT-RGB / EuroSAT-MS
#
# Fixes applied vs original:
#   FIX-1.  Removed channels_last everywhere — ViTs require contiguous layout.
#            (channels_last silently corrupts attention patterns → NaN loss)
#   FIX-2.  Moved torch.cuda.set_device() and DDP init to top of train(),
#            before model construction.  Original code tried to wrap `model`
#            in DDP before the variable existed.
#   FIX-3.  DINOHead last-layer init changed fill_(1) → fill_(0.1) to prevent
#            BF16 overflow on the first forward pass with proj_out=8192.
#   FIX-4.  student_temp raised 0.1 → 0.5; 0.1 causes log_softmax saturation
#            with large output dimensionality.
#   FIX-5.  teacher_temp_start/end raised (0.04/0.07 → 0.07/0.09) to match
#            values that are numerically stable with BF16.
#   FIX-6.  Added torch.isnan(loss) guard in training loop with informative
#            error rather than silently accumulating NaN.
#   FIX-7.  Corrected DINO.forward() to stay inside the existing no_grad block
#            for the teacher stream (nested context was redundant and fragile).
#   FIX-8.  clip_grad_norm_ moved inside scaler.unscale_() block — previously
#            clipping happened on scaled gradients (no-op / wrong scale).
#   FIX-9.  MultiCropCollator pin_memory buffers re-allocated per-batch if
#            batch size changes at the last batch (was silently returning
#            stale data for the final partial batch).
#   FIX-10. Added dist.init_process_group() call that was missing from train().
#
# Speed improvements (retained from original):
#   S2.  foreach EMA update (_foreach_mul_ / _foreach_add_)
#   S3.  Vectorized DINO cross-entropy loss
#   S4.  CUDA graph capture for inner training step (single-GPU only)
#   S5.  Gradient checkpointing on student backbone
#   S6.  kNN mode via scatter_add
#   S8.  torch.compile on student + teacher backbone (reduce-overhead)
#   S11. extract_features uses torch.inference_mode()
#   S12. Linear-probe eval uses frozen pre-extracted features
#   S13. DDP instead of DataParallel (~1.8× faster multi-GPU)
#   S14. Fused AdamW (single CUDA kernel for optimizer step)
#   S15. BF16 on Ampere+ (no loss scaler overhead)
#   S16. TF32 matrix multiplications (free on Ampere+)
#   S17. SDPA / FlashAttention via timm set_attn_backend
#   S18. Albumentations augmentation pipeline (2-4× faster than torchvision)
#   S19. proj_out 65536→8192, proj_hidden 2048→1024 (cheaper head)
#   S20. img_size 224→160 during pretraining (49% fewer ViT tokens)
#   S21. n_local 4→2 (25% less student compute)
#   S22. batch_size 128→512 (4× fewer steps/epoch)
#   S23. Compile DINOHead too
#   S24. deterministic=False (unlocks fused cuDNN kernels)
# =============================================================================

# !pip install timm torchvision torch scipy scikit-learn rasterio albumentations --quiet

import os, math, random, json, time, threading, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from torchvision.datasets.folder import default_loader
import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# ── Albumentations (optional, falls back to torchvision) ─────────────────────
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBU = True
except ImportError:
    HAS_ALBU = False
    print("albumentations not found — falling back to torchvision transforms.")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False   # S24
torch.backends.cudnn.benchmark     = True
torch.set_float32_matmul_precision("high")   # S16: TF32 on Ampere+

# ── Environment ───────────────────────────────────────────────────────────────
BASE_DIR    = "/kaggle/working"
DATA_DIR    = "/kaggle/input"
NUM_WORKERS = 4

COMPILE = torch.__version__ >= "2.0.0"

# ── S15: AMP dtype ────────────────────────────────────────────────────────────
def _amp_dtype():
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 \
           else torch.float16

AMP_DTYPE = _amp_dtype()

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    patternnet_dir  = f"{DATA_DIR}/datasets/samitsaleem/patternnet-scene-classification-dataset/PatternNet_Images",
    eurosat_rgb_dir = f"{DATA_DIR}/datasets/pranjallk1995/rgbeurosat/RBG",
    eurosat_ms_dir  = f"{DATA_DIR}/datasets/nguyenquangnhat2100/eurosatallbands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif",
    output_dir      = f"{BASE_DIR}/dino",
    checkpoint      = None,

    arch            = "vit_small_patch16_224",
    img_size        = 160,
    embed_dim       = 384,
    proj_hidden     = 1024,
    proj_out        = 8192,
    momentum        = 0.996,
    center_momentum = 0.9,

    n_global        = 2,
    n_local         = 2,
    local_size      = 96,
    local_scale     = (0.05, 0.32),
    global_scale    = (0.32, 1.0),

    epochs                     = 200,
    batch_size                 = 512,
    lr                         = 2e-3,
    min_lr                     = 1e-6,
    weight_decay_start         = 0.04,
    weight_decay_end           = 0.4,
    warmup_epochs              = 10,
    teacher_temp_warmup_epochs = 30,
    # FIX-5: raised temps for BF16 numerical stability (was 0.04 / 0.07)
    teacher_temp_start         = 0.07,
    teacher_temp_end           = 0.09,
    # FIX-4: raised student_temp to prevent log_softmax saturation (was 0.1)
    student_temp               = 0.5,

    jitter_strength   = 0.4,
    blur_prob_global1 = 1.0,
    blur_prob_global2 = 0.1,
    blur_prob_local   = 0.5,

    num_classes        = 10,
    knn_k              = 20,
    retrieval_ks       = [1, 5, 10],
    geo_thresholds_km  = [1, 5, 10],
    val_split          = 0.2,

    cuda_graph_warmup  = 3,
    grad_checkpoint    = True,
)

os.makedirs(CFG["output_dir"], exist_ok=True)


# =============================================================================
# ── Shared utilities ──────────────────────────────────────────────────────────
# =============================================================================

def _pw(nw): return nw > 0
def _pf(nw): return 4 if nw > 0 else None


def make_loader(ds, batch_size, shuffle, drop_last=False,
                collate_fn=None, sampler=None):
    kwargs = dict(
        batch_size         = batch_size,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        drop_last          = drop_last,
        persistent_workers = _pw(NUM_WORKERS),
        prefetch_factor    = _pf(NUM_WORKERS),
    )
    if sampler is not None:
        kwargs["sampler"] = sampler
    else:
        kwargs["shuffle"] = shuffle
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    return DataLoader(ds, **kwargs)


def ensure_split(root, val_frac=0.2, seed=SEED):
    if os.path.isdir(os.path.join(root, "train")):
        return root
    split_root = root.rstrip("/") + "_split"
    if os.path.isdir(os.path.join(split_root, "train")):
        print(f"  Using cached split at {split_root}")
        return split_root
    print(f"  Creating 80/20 stratified split → {split_root} …")
    base   = datasets.ImageFolder(root)
    labels = np.array([y for _, y in base.samples])
    sss    = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    for split_name, indices in [("train", train_idx), ("val", val_idx)]:
        for idx in indices:
            src_path, cls_idx = base.samples[idx]
            cls_name = base.classes[cls_idx]
            dst_dir  = os.path.join(split_root, split_name, cls_name)
            os.makedirs(dst_dir, exist_ok=True)
            dst_path = os.path.join(dst_dir, os.path.basename(src_path))
            if not os.path.exists(dst_path):
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)
    print(f"  Split complete: {len(train_idx)} train / {len(val_idx)} val")
    return split_root


# =============================================================================
# ── Augmentation pipeline ─────────────────────────────────────────────────────
# =============================================================================

def _base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        A.RandomResizedCrop(size=(size, size), scale=scale, interpolation=3),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=jitter_s, contrast=jitter_s,
                      saturation=jitter_s, hue=jitter_s * 0.25, p=0.8),
        A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(9, 9), sigma_limit=(0.1, 2.0), p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(A.Solarize(threshold=128, p=solarize_prob))
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


def _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        transforms.RandomResizedCrop(size, scale=scale, interpolation=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(
            brightness=jitter_s, contrast=jitter_s,
            saturation=jitter_s, hue=jitter_s * 0.25)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))], p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(transforms.RandomSolarize(threshold=128, p=solarize_prob))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


def _base_aug(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    return (_base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob)
            if HAS_ALBU else
            _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob))


# =============================================================================
# ── Datasets ──────────────────────────────────────────────────────────────────
# =============================================================================

class MultiCropDataset(Dataset):
    def __init__(self, root, cfg, cache=True):
        self.base      = datasets.ImageFolder(root)
        self.use_albu  = HAS_ALBU
        self.img_cache = {}

        self.global1_aug = _base_aug(cfg["img_size"], cfg["global_scale"],
                                      cfg["jitter_strength"], cfg["blur_prob_global1"])
        self.global2_aug = _base_aug(cfg["img_size"], cfg["global_scale"],
                                      cfg["jitter_strength"], cfg["blur_prob_global2"],
                                      solarize_prob=0.2)
        self.local_aug   = _base_aug(cfg["local_size"], cfg["local_scale"],
                                      cfg["jitter_strength"], cfg["blur_prob_local"])
        self.n_local     = cfg["n_local"]

        if cache:
            print(f"Caching {len(self.base)} images in RAM…", flush=True)
            for i, (path, _) in enumerate(self.base.samples):
                img = default_loader(path)
                self.img_cache[i] = np.array(img) if self.use_albu else img
                if (i + 1) % 5000 == 0:
                    print(f"  {i+1}/{len(self.base)} cached", flush=True)
            print("Cache complete.", flush=True)

    def __len__(self): return len(self.base)

    def _apply(self, aug, img):
        return aug(image=img)["image"] if self.use_albu else aug(img)

    def __getitem__(self, idx):
        if self.img_cache:
            img = self.img_cache[idx]
        else:
            raw = self.base[idx][0]
            img = np.array(raw) if self.use_albu else raw
        label = self.base.targets[idx]
        crops = [self._apply(self.global1_aug, img),
                 self._apply(self.global2_aug, img)]
        crops += [self._apply(self.local_aug, img) for _ in range(self.n_local)]
        return crops, label


# FIX-9: Re-allocate buffers if actual batch size differs (final partial batch)
class MultiCropCollator:
    def __init__(self, n_global, n_local, img_size, local_size, batch_size):
        self.views      = n_global + n_local
        self.n_global   = n_global
        self.n_local    = n_local
        self.img_size   = img_size
        self.local_size = local_size
        self._batch_size = batch_size
        self._bufs = self._make_bufs(batch_size)

    def _make_bufs(self, B):
        sizes = (
            [(B, 3, self.img_size,   self.img_size)]  * self.n_global +
            [(B, 3, self.local_size, self.local_size)] * self.n_local
        )
        return [torch.empty(*s).pin_memory() for s in sizes]

    def __call__(self, batch):
        B = len(batch)
        if B != self._batch_size:          # FIX-9: reallocate for partial batch
            self._bufs = self._make_bufs(B)
            self._batch_size = B
        for v in range(self.views):
            for b, (crops, _) in enumerate(batch):
                self._bufs[v][b] = crops[v]
        return [buf[:B] for buf in self._bufs], None


class TwoViewDataset(Dataset):
    def __init__(self, root, aug):
        self.base     = datasets.ImageFolder(root)
        self.aug      = aug
        self.use_albu = HAS_ALBU

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.use_albu:
            img = np.array(img)
            x1 = self.aug(image=img)["image"]
            x2 = self.aug(image=img)["image"]
        else:
            x1 = self.aug(img)
            x2 = self.aug(img)
        return x1, x2, label


# =============================================================================
# ── DINO projector head ───────────────────────────────────────────────────────
# =============================================================================

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=1024, bottleneck_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.last_layer = nn.utils.parametrizations.weight_norm(
            nn.Linear(bottleneck_dim, out_dim, bias=False))
        # FIX-3: fill_(1) → fill_(0.1) to prevent BF16 overflow with proj_out=8192
        self.last_layer.parametrizations.weight.original0.data.fill_(0.1)
        self.last_layer.parametrizations.weight.original0.requires_grad = False

    def forward(self, x):
        x = F.normalize(self.mlp(x), dim=-1, p=2)
        return self.last_layer(x)


# =============================================================================
# ── DINO model ────────────────────────────────────────────────────────────────
# =============================================================================

def _make_backbone(arch, img_size=160):
    m = timm.create_model(
        arch,
        pretrained=False,
        num_classes=0,
        img_size=img_size,
        dynamic_img_size=True,
    )
    if hasattr(m, "set_attn_backend"):
        try:
            m.set_attn_backend("sdpa")
        except Exception:
            pass
    return m


class DINO(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.n_global = cfg["n_global"]
        self.n_local  = cfg["n_local"]
        self.proj_out = cfg["proj_out"]

        self.student_backbone = _make_backbone(cfg["arch"], cfg["img_size"])
        self.student_head     = DINOHead(cfg["embed_dim"], cfg["proj_out"],
                                          cfg["proj_hidden"])

        self.teacher_backbone = _make_backbone(cfg["arch"], cfg["img_size"])
        self.teacher_head     = DINOHead(cfg["embed_dim"], cfg["proj_out"],
                                          cfg["proj_hidden"])
        for p in (list(self.teacher_backbone.parameters()) +
                  list(self.teacher_head.parameters())):
            p.requires_grad_(False)

        self.register_buffer("center", torch.zeros(1, cfg["proj_out"]))
        self.center_m = cfg["center_momentum"]

        # S5: gradient checkpointing on student backbone
        if cfg.get("grad_checkpoint") and hasattr(self.student_backbone,
                                                   "set_grad_checkpointing"):
            self.student_backbone.set_grad_checkpointing(True)

        # S2: cache param lists for foreach EMA
        self._t_bb_params = list(self.teacher_backbone.parameters())
        self._s_bb_params = list(self.student_backbone.parameters())
        self._t_hd_params = list(self.teacher_head.parameters())
        self._s_hd_params = list(self.student_head.parameters())

        # S23: compile heads
        if COMPILE:
            try:
                self.student_head = torch.compile(
                    self.student_head, mode="reduce-overhead")
                self.teacher_head = torch.compile(
                    self.teacher_head, mode="reduce-overhead")
            except Exception:
                pass

    # S2: foreach EMA
    @torch.no_grad()
    def _update_teacher(self, m):
        alpha = 1.0 - m
        torch._foreach_mul_(self._t_bb_params, m)
        torch._foreach_add_(self._t_bb_params, self._s_bb_params, alpha=alpha)
        torch._foreach_mul_(self._t_hd_params, m)
        torch._foreach_add_(self._t_hd_params, self._s_hd_params, alpha=alpha)

    @torch.no_grad()
    def _update_center(self, teacher_out):
        self.center.mul_(self.center_m).add_(
            teacher_out.mean(0, keepdim=True), alpha=1.0 - self.center_m)

    # S3: vectorized cross-entropy
    @staticmethod
    def _dino_loss_vectorized(student_chunks, teacher_chunks, student_temp):
        G       = len(teacher_chunks)
        V       = len(student_chunks)
        t_stack = torch.stack(teacher_chunks)                       # (G, B, D)
        s_stack = torch.stack(student_chunks)                       # (V, B, D)
        s_log   = F.log_softmax(s_stack / student_temp, dim=-1)     # (V, B, D)
        total   = torch.tensor(0.0, device=t_stack.device)
        n_terms = 0
        for t_idx in range(G):
            ce = -(t_stack[t_idx].unsqueeze(0) * s_log).sum(-1)    # (V, B)
            for s_idx in range(V):
                if s_idx == t_idx:
                    continue
                total   = total + ce[s_idx].mean()
                n_terms += 1
        return total / n_terms

    def forward(self, crops, student_temp, teacher_temp, momentum):
        n_global = self.n_global
        n_crops  = len(crops)

        global_crops = crops[:n_global]
        local_crops  = crops[n_global:]

        # Student: two passes grouped by spatial size
        # FIX-1: crops arrive as plain contiguous tensors (no channels_last)
        global_feats = self.student_backbone(torch.cat(global_crops, dim=0))
        global_out   = self.student_head(global_feats)
        local_feats  = self.student_backbone(torch.cat(local_crops,  dim=0))
        local_out    = self.student_head(local_feats)

        student_out = torch.cat([global_out, local_out], dim=0).chunk(n_crops)

        # FIX-7: teacher entirely inside a single no_grad block (no nested ctx)
        with torch.no_grad():
            self._update_teacher(momentum)
            teacher_feats  = self.teacher_backbone(torch.cat(global_crops, dim=0))
            teacher_out    = self.teacher_head(teacher_feats)
            teacher_out    = F.softmax(
                (teacher_out - self.center) / teacher_temp, dim=-1)
            teacher_chunks = teacher_out.chunk(n_global)
            self._update_center(teacher_out)

        return self._dino_loss_vectorized(
            list(student_out), list(teacher_chunks), student_temp)


# =============================================================================
# ── Schedule helpers ──────────────────────────────────────────────────────────
# =============================================================================

def cosine_schedule(start, end, epoch, total):
    return end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * epoch / total))


def build_epoch_schedules(cfg):
    epochs          = cfg["epochs"]
    lrs, wds, momenta, t_temps = [], [], [], []
    for e in range(epochs):
        if e < cfg["warmup_epochs"]:
            lr = cfg["lr"] * (e + 1) / cfg["warmup_epochs"]
        else:
            lr = cosine_schedule(cfg["lr"], cfg["min_lr"],
                                 e - cfg["warmup_epochs"],
                                 epochs - cfg["warmup_epochs"])
        wd  = cosine_schedule(cfg["weight_decay_start"],
                               cfg["weight_decay_end"], e, epochs)
        mom = cosine_schedule(cfg["momentum"], 1.0, e, epochs)
        tt  = cosine_schedule(cfg["teacher_temp_start"],
                               cfg["teacher_temp_end"],
                               min(e, cfg["teacher_temp_warmup_epochs"]),
                               cfg["teacher_temp_warmup_epochs"])
        lrs.append(lr); wds.append(wd)
        momenta.append(mom); t_temps.append(tt)
    return lrs, wds, momenta, t_temps


def apply_lr_wd(optimizer, lr, wd):
    for g in optimizer.param_groups:
        g["lr"] = lr
        if g.get("apply_wd", True):
            g["weight_decay"] = wd


# =============================================================================
# ── Async checkpoint saver ────────────────────────────────────────────────────
# =============================================================================

_save_thread: threading.Thread = None

def async_save(path, obj):
    global _save_thread
    if _save_thread is not None:
        _save_thread.join()
    def _save():
        torch.save(obj, path)
        print(f"  → Saved {path}", flush=True)
    _save_thread = threading.Thread(target=_save, daemon=True)
    _save_thread.start()


# =============================================================================
# ── Training loop (DDP-aware) ─────────────────────────────────────────────────
# =============================================================================

def train(rank: int, world_size: int):
    # FIX-2: set device and init process group FIRST, before anything else
    torch.cuda.set_device(rank)
    device  = torch.device(f"cuda:{rank}")
    is_ddp  = world_size > 1
    is_main = rank == 0

    # FIX-10: init_process_group was missing entirely in the original
    if is_ddp:
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            world_size=world_size,
            rank=rank,
        )

    if is_main:
        print(f"Device: {device}  |  World: {world_size}  |  "
              f"Workers: {NUM_WORKERS}  |  compile: {COMPILE}  |  "
              f"AMP: {AMP_DTYPE}", flush=True)

    # ── Data ──────────────────────────────────────────────────────────────────
    dataset  = MultiCropDataset(CFG["patternnet_dir"], CFG, cache=True)
    sampler  = (DistributedSampler(dataset, num_replicas=world_size,
                                    rank=rank, shuffle=True, drop_last=True)
                if is_ddp else None)
    collator = MultiCropCollator(CFG["n_global"], CFG["n_local"],
                                  CFG["img_size"], CFG["local_size"],
                                  CFG["batch_size"])
    loader   = make_loader(dataset, CFG["batch_size"],
                           shuffle=not is_ddp, drop_last=True,
                           collate_fn=collator, sampler=sampler)

    # ── Model ─────────────────────────────────────────────────────────────────
    # FIX-1: no .to(memory_format=torch.channels_last) — ViTs need contiguous
    model = DINO(CFG).to(device)

    # S8: compile backbones before DDP wrapping
    if COMPILE:
        try:
            model.student_backbone = torch.compile(
                model.student_backbone, mode="reduce-overhead")
            model.teacher_backbone = torch.compile(
                model.teacher_backbone, mode="reduce-overhead")
            if is_main:
                print("torch.compile enabled on student + teacher backbone.")
        except Exception as e:
            if is_main:
                print(f"torch.compile skipped: {e}")

    # FIX-2 / S13: DDP wrapping happens AFTER model is constructed
    if is_ddp:
        model.student_backbone = DDP(model.student_backbone,
                                      device_ids=[rank],
                                      find_unused_parameters=False)
        model.student_head     = DDP(model.student_head,
                                      device_ids=[rank],
                                      find_unused_parameters=False)

    # ── Optimizer ─────────────────────────────────────────────────────────────
    wd_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and "bias" not in n and "norm" not in n]
    no_wd     = [p for n, p in model.named_parameters()
                 if p.requires_grad and ("bias" in n or "norm" in n)]

    # S14: fused AdamW
    optimizer = torch.optim.AdamW(
        [{"params": wd_params, "apply_wd": True},
         {"params": no_wd,     "apply_wd": False, "weight_decay": 0.0}],
        lr=CFG["lr"], weight_decay=CFG["weight_decay_start"],
        fused=True,
    )

    use_amp    = AMP_DTYPE is not None
    use_scaler = AMP_DTYPE == torch.float16
    scaler     = torch.amp.GradScaler("cuda", enabled=use_scaler)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    lrs, wds, momenta, t_temps = build_epoch_schedules(CFG)

    start_epoch = 0
    if CFG["checkpoint"] and is_main:
        ckpt = torch.load(CFG["checkpoint"], map_location="cpu")
        model.load_state_dict(ckpt["model"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {start_epoch}")

    # ── S4: CUDA graph (single-GPU, no compile — compile already optimises) ───
    use_graph = (
        not is_ddp
        and not COMPILE
        and hasattr(torch.cuda, "CUDAGraph")
    )
    graph        = None
    static_crops = None
    static_loss  = None
    warmup_done  = False
    step_counter = 0
    warmup_steps = CFG.get("cuda_graph_warmup", 3)

    log           = []
    t_train_start = time.time()

    for epoch in range(start_epoch, CFG["epochs"]):
        if is_ddp:
            sampler.set_epoch(epoch)

        lr           = lrs[epoch]
        wd           = wds[epoch]
        momentum     = momenta[epoch]
        teacher_temp = t_temps[epoch]
        apply_lr_wd(optimizer, lr, wd)

        model.train()
        total_loss = 0.0
        t0         = time.time()

        for crops, _ in loader:
            # FIX-1: plain .to(device) — no channels_last for ViT
            crops = [c.to(device, non_blocking=True) for c in crops]

            # ── CUDA graph replay ─────────────────────────────────────────────
            if use_graph and warmup_done and graph is not None:
                for i, c in enumerate(crops):
                    static_crops[i].copy_(c)
                optimizer.zero_grad(set_to_none=True)
                graph.replay()
                scaler.unscale_(optimizer)
                # FIX-8: clip_grad_norm_ must be inside unscale_() block
                nn.utils.clip_grad_norm_(trainable_params, max_norm=3.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += static_loss.item()
                step_counter += 1
                continue

            # ── Normal step ───────────────────────────────────────────────────
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                loss = model(crops, CFG["student_temp"], teacher_temp, momentum)
                if isinstance(loss, torch.Tensor) and loss.dim() > 0:
                    loss = loss.mean()

            # FIX-6: guard against NaN before it accumulates silently
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[rank {rank}] NaN/Inf loss at step {step_counter}, "
                      f"epoch {epoch+1}. Skipping batch.", flush=True)
                optimizer.zero_grad(set_to_none=True)
                step_counter += 1
                continue

            scaler.scale(loss).backward()
            # FIX-8: unscale_ before clip so the norm is on true gradient scale
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(trainable_params, max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            step_counter += 1

            # Capture CUDA graph after warm-up
            if use_graph and not warmup_done and step_counter == warmup_steps:
                torch.cuda.synchronize()
                static_crops = [c.clone() for c in crops]
                static_loss  = torch.zeros(1, device=device)
                g = torch.cuda.CUDAGraph()
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.graph(g):
                    with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                        _loss = model(static_crops, CFG["student_temp"],
                                      teacher_temp, momentum)
                        if isinstance(_loss, torch.Tensor) and _loss.dim() > 0:
                            _loss = _loss.mean()
                    static_loss.copy_(_loss)
                    scaler.scale(_loss).backward()
                graph       = g
                warmup_done = True
                if is_main:
                    print(f"  CUDA graph captured at step {step_counter}.")

        avg_loss = total_loss / max(len(loader), 1)
        elapsed  = time.time() - t0

        if is_main:
            log.append({"epoch": epoch, "loss": avg_loss, "lr": lr,
                        "teacher_temp": teacher_temp, "momentum": momentum,
                        "epoch_time_s": round(elapsed, 1)})
            print(f"Epoch [{epoch+1:>3}/{CFG['epochs']}]  "
                  f"loss={avg_loss:.4f}  lr={lr:.2e}  "
                  f"t_temp={teacher_temp:.4f}  time={elapsed:.0f}s")

            if (epoch + 1) % 50 == 0 or epoch == CFG["epochs"] - 1:
                raw   = model
                state = {k.replace("module.", ""): v
                         for k, v in raw.state_dict().items()}
                path  = os.path.join(CFG["output_dir"], f"dino_ep{epoch+1}.pt")
                async_save(path, {"epoch": epoch, "model": state,
                                   "optimizer": optimizer.state_dict(),
                                   "cfg": CFG})

    if is_main:
        if _save_thread is not None:
            _save_thread.join()
        with open(os.path.join(CFG["output_dir"], "dino_log.json"), "w") as f:
            json.dump(log, f, indent=2)
        total = time.time() - t_train_start
        print(f"\nPre-training done.  Total: {total/3600:.2f} h ({total:.0f} s)")

    if is_ddp:
        dist.destroy_process_group()


# =============================================================================
# ── Shared eval helpers ───────────────────────────────────────────────────────
# =============================================================================

def load_backbone(backbone_path, device):
    ckpt  = torch.load(backbone_path, map_location="cpu")
    state = {k.replace("module.", "").replace("student_backbone.", ""): v
             for k, v in ckpt["model"].items() if "student_backbone" in k}
    backbone = _make_backbone(CFG["arch"])
    backbone.load_state_dict(state, strict=False)
    # FIX-1: no channels_last for ViT backbone
    backbone.eval().to(device)
    for p in backbone.parameters():
        p.requires_grad_(False)
    return backbone


def get_eval_transforms():
    val_tf = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


@torch.inference_mode()
def extract_features(backbone, loader, device):
    all_feats, all_labels = [], []
    backbone.eval()
    use_amp = AMP_DTYPE is not None
    for x, y in loader:
        # FIX-1: plain .to(device) for ViT
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
            feats = backbone(x.to(device, non_blocking=True))
        all_feats.append(feats.float().cpu())
        all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


# =============================================================================
# §4.1  CLASSIFICATION — linear probe at multiple label fractions
# =============================================================================

def eval_classification(backbone_path, eurosat_dir,
                         label_fracs=(0.01, 0.1, 1.0), num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    train_tf, val_tf = get_eval_transforms()
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])

    train_full = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=train_tf)
    val_ds     = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)

    print("  Pre-extracting features for linear probe…")
    all_train_feats, all_train_labels = extract_features(
        backbone, make_loader(train_full, 512, shuffle=False), device)
    val_feats, val_labels = extract_features(
        backbone, make_loader(val_ds, 512, shuffle=False), device)

    all_train_feats  = all_train_feats.to(device)
    all_train_labels = all_train_labels.to(device)
    val_feats_dev    = val_feats.to(device)
    val_labels_dev   = val_labels.to(device)

    results = {}
    bs      = 256
    for frac in label_fracs:
        n       = max(num_classes, int(len(train_full) * frac))
        indices = random.sample(range(len(all_train_feats)), n)
        idx_t   = torch.tensor(indices, device=device)
        f_sub   = all_train_feats[idx_t]
        l_sub   = all_train_labels[idx_t]

        head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
        opt   = torch.optim.SGD(head.parameters(), lr=0.1,
                                 momentum=0.9, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

        for _ in range(100):
            head.train()
            perm = torch.randperm(len(f_sub), device=device)
            for start in range(0, len(f_sub), bs):
                sel = perm[start:start + bs]
                opt.zero_grad()
                F.cross_entropy(head(f_sub[sel]), l_sub[sel]).backward()
                opt.step()
            sched.step()

        head.eval()
        with torch.no_grad():
            preds_all = [head(val_feats_dev[s:s + bs]).argmax(1)
                         for s in range(0, len(val_feats_dev), bs)]
        preds  = torch.cat(preds_all).cpu().numpy()
        labels = val_labels_dev.cpu().numpy()

        acc      = 100.0 * (preds == labels).mean()
        macro_f1 = 100.0 * f1_score(labels, preds, average="macro")
        key = f"{int(round(frac * 100))}pct"
        results[key] = {"top1_acc": round(acc, 2), "macro_f1": round(macro_f1, 2)}
        print(f"  [{int(frac*100)}% labels]  "
              f"Top-1={acc:.2f}%  Macro-F1={macro_f1:.2f}%")

    return results


# =============================================================================
# §4.2  SEGMENTATION — frozen backbone + linear pixel decoder
# =============================================================================

class SegDecoder(nn.Module):
    def __init__(self, embed_dim, num_classes, patch_size=16, img_size=224):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.head      = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, patch_tokens):
        B, N, D = patch_tokens.shape
        g = self.grid_size
        x = patch_tokens.permute(0, 2, 1).reshape(B, D, g, g)
        x = self.head(x)
        return F.interpolate(x, size=(224, 224), mode="bilinear",
                             align_corners=False)


def boundary_f1(pred_mask, gt_mask, num_classes, dilation=1):
    from scipy.ndimage import binary_dilation as bd
    bf1_list = []
    for c in range(num_classes):
        p = (pred_mask == c).astype(np.uint8)
        g = (gt_mask   == c).astype(np.uint8)
        if g.sum() == 0: continue
        p_b   = np.logical_xor(p, bd(p, iterations=dilation)).astype(np.uint8)
        g_b   = np.logical_xor(g, bd(g, iterations=dilation)).astype(np.uint8)
        inter = (p_b & g_b).sum(); denom = p_b.sum() + g_b.sum()
        if denom == 0: continue
        bf1_list.append(2 * inter / (denom + 1e-8))
    return float(np.mean(bf1_list)) if bf1_list else 0.0


def eval_segmentation(backbone_path, eurosat_dir, num_classes=10, epochs=30):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone  = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()
    aug_tf    = transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=aug_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 64, shuffle=True,  drop_last=True)
    val_ldr   = make_loader(val_ds,   64, shuffle=False)

    decoder = SegDecoder(CFG["embed_dim"], num_classes).to(device)
    opt     = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = AMP_DTYPE is not None

    def get_patch_tokens(x):
        with torch.inference_mode():
            out = backbone.forward_features(
                x.to(device, non_blocking=True))   # FIX-1: no channels_last
            if isinstance(out, torch.Tensor) and out.dim() == 3:
                return out[:, 1:]
            return backbone(x).unsqueeze(1).expand(-1, 196, -1)

    for epoch in range(epochs):
        decoder.train()
        for x, y in train_ldr:
            x, y = (x.to(device, non_blocking=True),
                    y.to(device, non_blocking=True))
            tokens = get_patch_tokens(x)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                logits = decoder(tokens)
                target = y.view(-1, 1, 1).expand(-1, 224, 224)
                loss   = F.cross_entropy(logits, target)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Seg epoch {epoch+1}/{epochs}", flush=True)

    decoder.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    all_bf1   = []
    with torch.inference_mode():
        for x, y in val_ldr:
            tokens = get_patch_tokens(x.to(device, non_blocking=True))
            pred   = decoder(tokens).argmax(1).cpu().numpy()
            label  = y.numpy()
            for b in range(pred.shape[0]):
                gt_map = np.full_like(pred[b], label[b])
                for i in range(num_classes):
                    for j in range(num_classes):
                        confusion[i, j] += (
                            (gt_map == i) & (pred[b] == j)).sum()
                all_bf1.append(boundary_f1(pred[b], gt_map, num_classes))

    iou_per_class = []
    for c in range(num_classes):
        tp    = confusion[c, c]
        fp    = confusion[:, c].sum() - tp
        fn    = confusion[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class.append(tp / denom)

    miou     = float(np.mean(iou_per_class)) * 100
    mean_bf1 = float(np.mean(all_bf1)) * 100
    print(f"  Segmentation  mIoU={miou:.2f}%  Boundary-F1={mean_bf1:.2f}%")
    return {"miou": round(miou, 2), "boundary_f1": round(mean_bf1, 2)}


# =============================================================================
# §4.3  RETRIEVAL PERFORMANCE
# =============================================================================

def haversine_km(lat1, lon1, lat2, lon2):
    R  = 6371.0
    dr = math.radians
    dlat = dr(lat2 - lat1); dlon = dr(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(dr(lat1)) * math.cos(dr(lat2)) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def mean_average_precision(sim_matrix, labels):
    N  = sim_matrix.shape[0]
    sm = sim_matrix.clone()
    sm.fill_diagonal_(-1e9)
    order   = sm.argsort(dim=1, descending=True)
    ap_list = []
    for i in range(N):
        gt    = (labels[order[i]] == labels[i])
        n_rel = gt.sum().item()
        if n_rel == 0: continue
        n_correct = 0; precisions = []
        for rank, hit in enumerate(gt.tolist(), 1):
            if hit:
                n_correct += 1
                precisions.append(n_correct / rank)
        ap_list.append(sum(precisions) / n_rel)
    return float(np.mean(ap_list)) * 100 if ap_list else 0.0


def eval_retrieval(backbone_path, eurosat_dir, gps_csv=None,
                    ks=(1, 5, 10), geo_thresholds_km=(1, 5, 10)):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone  = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)

    print("  Extracting gallery features…")
    feats, labels = extract_features(backbone, val_ldr, device)
    feats = F.normalize(feats.float(), dim=-1)
    sim   = feats @ feats.T
    N     = feats.shape[0]

    results     = {}
    sim_no_diag = sim.clone()
    sim_no_diag.fill_diagonal_(-1e9)
    topk_max    = max(ks)
    top_indices = sim_no_diag.topk(topk_max, dim=1).indices

    for k in ks:
        top_k_labels = labels[top_indices[:, :k]]
        correct = (top_k_labels == labels.unsqueeze(1)).any(dim=1).sum().item()
        recall  = 100.0 * correct / N
        results[f"recall@{k}"] = round(recall, 2)
        print(f"  Recall@{k} = {recall:.2f}%")

    mAP = mean_average_precision(sim, labels)
    results["mAP"] = round(mAP, 2)
    print(f"  mAP = {mAP:.2f}%")

    if gps_csv is not None:
        import pandas as pd
        gps_df       = pd.read_csv(gps_csv)
        img_paths    = [val_ds.samples[i][0] for i in range(N)]
        fname_to_gps = {row["filename"]: (row["lat"], row["lon"])
                        for _, row in gps_df.iterrows()}
        errors_km = []
        geo_hits  = {k: {eps: 0 for eps in geo_thresholds_km} for k in ks}
        n_valid   = 0

        for i in range(N):
            q_fname = os.path.basename(img_paths[i])
            if q_fname not in fname_to_gps: continue
            n_valid += 1
            q_lat, q_lon = fname_to_gps[q_fname]
            top_fnames   = [os.path.basename(img_paths[j])
                            for j in top_indices[i, :topk_max].tolist()]
            if top_fnames[0] in fname_to_gps:
                r_lat, r_lon = fname_to_gps[top_fnames[0]]
                errors_km.append(haversine_km(q_lat, q_lon, r_lat, r_lon))
            for k in ks:
                cands = [f for f in top_fnames[:k] if f in fname_to_gps]
                for eps in geo_thresholds_km:
                    if any(haversine_km(q_lat, q_lon, *fname_to_gps[f]) <= eps
                           for f in cands):
                        geo_hits[k][eps] += 1

        if errors_km:
            med_err = float(np.median(errors_km)) * 1000
            p90_err = float(np.percentile(errors_km, 90)) * 1000
            results["median_loc_error_m"] = round(med_err, 1)
            results["p90_loc_error_m"]    = round(p90_err, 1)
            print(f"  Median loc. error = {med_err:.1f} m  "
                  f"(P90 = {p90_err:.1f} m)")

        if n_valid > 0:
            for k in ks:
                for eps in geo_thresholds_km:
                    r = 100.0 * geo_hits[k][eps] / n_valid
                    results[f"geo_recall@{k}_{eps}km"] = round(r, 2)
                    print(f"  Geo-Recall@{k} ({eps} km) = {r:.2f}%")

    return results


# =============================================================================
# §4.4  REPRESENTATION QUALITY
# =============================================================================

def effective_rank(feats):
    f       = feats - feats.mean(0)
    cov     = (f.T @ f) / (feats.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov.float()).clamp(min=0)
    eigvals = eigvals / eigvals.sum().clamp(min=1e-8)
    eigvals = eigvals[eigvals > 1e-9]
    entropy = -(eigvals * eigvals.log()).sum()
    return math.exp(entropy.item())


def uniformity_score(feats):
    feats = F.normalize(feats.float(), dim=-1)
    if feats.shape[0] > 2000:
        feats = feats[torch.randperm(feats.shape[0])[:2000]]
    sq = torch.cdist(feats, feats, p=2).pow(2)
    return round(sq.mul(-2).exp().mean().log().item(), 4)


def batched_knn_predict(sim, train_labels, k, num_classes, chunk=512):
    N_val  = sim.shape[0]
    device = sim.device
    preds  = torch.empty(N_val, dtype=torch.long, device=device)
    for start in range(0, N_val, chunk):
        end        = min(start + chunk, N_val)
        top_idx    = sim[start:end].topk(k, dim=1).indices
        top_labels = train_labels.to(device)[top_idx]
        B          = end - start
        votes      = torch.zeros(B, num_classes, device=device)
        votes.scatter_add_(1, top_labels,
                           torch.ones(B, k, device=device))
        preds[start:end] = votes.argmax(1)
    return preds


def eval_representation_quality(backbone_path, eurosat_dir,
                                  knn_k=20, num_classes=10):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone  = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=val_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 256, shuffle=False)
    val_ldr   = make_loader(val_ds,   256, shuffle=False)

    print("  Extracting features for representation quality…")
    train_feats, train_labels = extract_features(backbone, train_ldr, device)
    val_feats,   val_labels   = extract_features(backbone, val_ldr,   device)
    train_n = F.normalize(train_feats.float(), dim=-1)
    val_n   = F.normalize(val_feats.float(),   dim=-1)

    sim       = val_n @ train_n.T
    knn_preds = batched_knn_predict(sim, train_labels, knn_k, num_classes)
    knn_acc   = 100.0 * (knn_preds.cpu() == val_labels).float().mean().item()
    print(f"  kNN accuracy (k={knn_k}) = {knn_acc:.2f}%")

    eff_rank = effective_rank(val_feats.float())
    print(f"  Effective rank = {eff_rank:.1f}")

    unif = uniformity_score(val_n)
    print(f"  Uniformity = {unif:.4f}")

    aug = _base_aug(CFG["img_size"], CFG["global_scale"],
                    CFG["jitter_strength"], CFG["blur_prob_global1"])
    two_view_ds  = TwoViewDataset(os.path.join(split_dir, "val"), aug)
    two_view_ldr = make_loader(two_view_ds, 256, shuffle=False)
    align_scores = []
    with torch.inference_mode():
        for x1, x2, _ in two_view_ldr:
            # FIX-1: no channels_last
            z1 = F.normalize(
                backbone(x1.to(device, non_blocking=True)), dim=-1)
            z2 = F.normalize(
                backbone(x2.to(device, non_blocking=True)), dim=-1)
            align_scores.append(
                (z1 - z2).pow(2).sum(dim=-1).mean().item())
    alignment = round(float(np.mean(align_scores)), 4)
    print(f"  Alignment = {alignment:.4f}")

    return {"knn_acc":        round(knn_acc, 2),
            "effective_rank": round(eff_rank, 1),
            "uniformity":     unif,
            "alignment":      alignment}


# =============================================================================
# §4.5  BAND MISMATCH ROBUSTNESS
# =============================================================================

class BandAdapterBackbone(nn.Module):
    def __init__(self, backbone, in_channels=13, out_channels=3):
        super().__init__()
        self.adapter  = nn.Conv2d(in_channels, out_channels,
                                   kernel_size=1, bias=False)
        self.backbone = backbone
        nn.init.kaiming_normal_(self.adapter.weight)

    def forward(self, x):
        return self.backbone(self.adapter(x))


class MultiSpectralDataset(Dataset):
    def __init__(self, root, n_channels=13):
        self.samples    = []
        self.n_channels = n_channels
        classes = sorted(d for d in os.listdir(root)
                         if os.path.isdir(os.path.join(root, d)))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(
                        (".tif", ".tiff", ".npy", ".png", ".jpg")):
                    self.samples.append(
                        (os.path.join(cls_dir, fname),
                         self.class_to_idx[cls]))

    def __len__(self): return len(self.samples)

    def _load_tif(self, path):
        try:
            import rasterio
        except ImportError:
            raise ImportError("pip install rasterio")
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)
        n = self.n_channels
        if arr.shape[0] >= n:
            arr = arr[:n]
        else:
            pad = np.zeros((n - arr.shape[0], *arr.shape[1:]),
                           dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        return torch.from_numpy(arr)

    def _normalise(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.view(x.shape[0], -1).mean(1, keepdim=True).unsqueeze(-1)
        std  = (x.view(x.shape[0], -1).std(1, keepdim=True)
                 .unsqueeze(-1).clamp(min=1e-6))
        return (x - mean) / std

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        ext = os.path.splitext(path)[1].lower()

        if ext in (".tif", ".tiff"):
            x = self._load_tif(path)
        elif ext == ".npy":
            arr = np.load(path).astype(np.float32)
            if arr.ndim == 3 and arr.shape[2] == self.n_channels:
                arr = arr.transpose(2, 0, 1)
            x = torch.from_numpy(arr)
            if x.shape[0] > self.n_channels:
                x = x[:self.n_channels]
        else:
            from PIL import Image
            img = Image.open(path).convert("RGB")
            x   = transforms.Compose([
                transforms.Resize(256), transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                      [0.229, 0.224, 0.225]),
            ])(img)
            x = x.repeat(5, 1, 1)[:self.n_channels]
            return x, label

        x = self._normalise(x)
        x = F.interpolate(x[None], size=224, mode="bilinear",
                          align_corners=False)[0]
        return x, label


def eval_band_mismatch(backbone_path, eurosat_rgb_dir,
                        eurosat_ms_dir=None, num_classes=10):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone  = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(backbone, val_ldr, device)
    feats  = F.normalize(feats.float(), dim=-1)
    sim_nd = (feats @ feats.T)
    sim_nd.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim_nd.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13)
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb   = BandAdapterBackbone(
                backbone, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)

            tmp_head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()), lr=1e-3)

            for _ in range(5):
                adapter_bb.adapter.train(); tmp_head.train()
                for x, y in ms_ldr_train:
                    x, y = (x.to(device, non_blocking=True),
                            y.to(device, non_blocking=True))
                    with torch.inference_mode():
                        feat = backbone(adapter_bb.adapter(x))
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.inference_mode():
                for x, y in ms_ldr_eval:
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True))
                        .float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels
            ).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms is not None
                               else None,
        "band_mismatch_delta": delta,
    }


# =============================================================================
# ── Full evaluation orchestrator ──────────────────────────────────────────────
# =============================================================================

def run_full_evaluation(backbone_path, eurosat_rgb_dir=None,
                         eurosat_ms_dir=None, gps_csv=None):
    eurosat_rgb_dir = eurosat_rgb_dir or CFG["eurosat_rgb_dir"]
    eurosat_ms_dir  = eurosat_ms_dir  or CFG.get("eurosat_ms_dir")
    all_results     = {"model": "DINO", "checkpoint": backbone_path}

    print("\n" + "=" * 60)
    print("§4.1  CLASSIFICATION (linear probe)")
    print("=" * 60)
    all_results["classification"] = eval_classification(
        backbone_path, eurosat_rgb_dir,
        label_fracs=(0.01, 0.1, 1.0), num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.2  SEGMENTATION")
    print("=" * 60)
    all_results["segmentation"] = eval_segmentation(
        backbone_path, eurosat_rgb_dir, num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.3  RETRIEVAL PERFORMANCE")
    print("=" * 60)
    all_results["retrieval"] = eval_retrieval(
        backbone_path, eurosat_rgb_dir, gps_csv=gps_csv,
        ks=CFG["retrieval_ks"], geo_thresholds_km=CFG["geo_thresholds_km"])

    print("\n" + "=" * 60)
    print("§4.4  REPRESENTATION QUALITY")
    print("=" * 60)
    all_results["representation"] = eval_representation_quality(
        backbone_path, eurosat_rgb_dir,
        knn_k=CFG["knn_k"], num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.5  BAND MISMATCH ROBUSTNESS")
    print("=" * 60)
    all_results["band_mismatch"] = eval_band_mismatch(
        backbone_path, eurosat_rgb_dir, eurosat_ms_dir)

    out_path = os.path.join(CFG["output_dir"], "dino_eval_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nAll results saved to {out_path}")
    return all_results


# =============================================================================
# ── Entry point ───────────────────────────────────────────────────────────────
# =============================================================================

def is_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

if __name__ == "__main__":
    world_size = torch.cuda.device_count()

    if world_size > 1 and not is_notebook():
        os.environ["MASTER_ADDR"] = "localhost"
        os.environ["MASTER_PORT"] = "12355"
        mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)
    else:
        if world_size > 1 and is_notebook():
            print(f"Notebook detected — DDP disabled. "
                  f"Running single-GPU on cuda:0. "
                  f"({world_size} GPUs available but only 1 will be used.)")
        train(rank=0, world_size=1)

    BEST_CKPT = os.path.join(CFG["output_dir"], f"dino_ep{CFG['epochs']}.pt")
    GPS_CSV   = f"{DATA_DIR}/eurosat/eurosat_gps.csv"
    run_full_evaluation(
        BEST_CKPT,
        CFG["eurosat_rgb_dir"],
        CFG["eurosat_ms_dir"],
        GPS_CSV if os.path.exists(GPS_CSV) else None,
    )

Notebook detected — DDP disabled. Running single-GPU on cuda:0. (2 GPUs available but only 1 will be used.)
Device: cuda:0  |  World: 1  |  Workers: 4  |  compile: True  |  AMP: torch.float16
Caching 30399 images in RAM…


/tmp/ipykernel_441/3009048983.py:224: UserWarning: Argument(s) 'threshold' are not valid for transform Solarize
  ops.append(A.Solarize(threshold=128, p=solarize_prob))


  5000/30399 cached
  10000/30399 cached
  15000/30399 cached
  20000/30399 cached
  25000/30399 cached
  30000/30399 cached
Cache complete.
torch.compile enabled on student + teacher backbone.


W0417 09:53:36.410000 441 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch [  1/200]  loss=9.0098  lr=2.00e-04  t_temp=0.0700  time=222s
Epoch [  2/200]  loss=9.0061  lr=4.00e-04  t_temp=0.0701  time=140s
Epoch [  3/200]  loss=8.9753  lr=6.00e-04  t_temp=0.0702  time=138s
Epoch [  4/200]  loss=8.9074  lr=8.00e-04  t_temp=0.0705  time=137s
Epoch [  5/200]  loss=8.8585  lr=1.00e-03  t_temp=0.0709  time=137s
Epoch [  6/200]  loss=8.8370  lr=1.20e-03  t_temp=0.0713  time=137s
Epoch [  7/200]  loss=8.8287  lr=1.40e-03  t_temp=0.0719  time=136s
Epoch [  8/200]  loss=8.8257  lr=1.60e-03  t_temp=0.0726  time=137s
Epoch [  9/200]  loss=8.8249  lr=1.80e-03  t_temp=0.0733  time=136s
Epoch [ 10/200]  loss=8.8250  lr=2.00e-03  t_temp=0.0741  time=136s
Epoch [ 11/200]  loss=8.8257  lr=2.00e-03  t_temp=0.0750  time=135s
Epoch [ 12/200]  loss=8.8266  lr=2.00e-03  t_temp=0.0759  time=136s
Epoch [ 13/200]  loss=8.8277  lr=2.00e-03  t_temp=0.0769  time=135s
Epoch [ 14/200]  loss=8.8289  lr=2.00e-03  t_temp=0.0779  time=136s
Epoch [ 15/200]  loss=8.8302  lr=2.00e-03  t_tem

RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [2]:
# ── Cell: Self-contained §4.5 eval after kernel reset ────────────────────────
# Run this as a single cell. It reimports everything, redefines CFG and all
# helpers, then runs the fixed eval_band_mismatch.

import os, math, random, json, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import timm
from sklearn.model_selection import StratifiedShuffleSplit

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBU = True
except ImportError:
    HAS_ALBU = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_DIR    = "/kaggle/working"
DATA_DIR    = "/kaggle/input"
NUM_WORKERS = 4

def _amp_dtype():
    if not torch.cuda.is_available(): return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 \
           else torch.float16
AMP_DTYPE = _amp_dtype()

CFG = dict(
    patternnet_dir  = f"{DATA_DIR}/datasets/samitsaleem/patternnet-scene-classification-dataset/PatternNet_Images",
    eurosat_rgb_dir = f"{DATA_DIR}/datasets/pranjallk1995/rgbeurosat/RBG",
    eurosat_ms_dir  = f"{DATA_DIR}/datasets/nguyenquangnhat2100/eurosatallbands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif",
    output_dir      = f"{BASE_DIR}/dino",
    arch            = "vit_small_patch16_224",
    img_size        = 160,
    embed_dim       = 384,
    proj_hidden     = 1024,
    proj_out        = 8192,
    epochs          = 200,
    num_classes     = 10,
    knn_k           = 20,
    val_split       = 0.2,
    retrieval_ks       = [1, 5, 10],
    geo_thresholds_km  = [1, 5, 10],
    global_scale       = (0.32, 1.0),
    jitter_strength    = 0.4,
    blur_prob_global1  = 1.0,
)

# ── Helpers ───────────────────────────────────────────────────────────────────

def make_loader(ds, batch_size, shuffle, drop_last=False, sampler=None):
    pw = NUM_WORKERS > 0
    kwargs = dict(batch_size=batch_size, num_workers=NUM_WORKERS,
                  pin_memory=True, drop_last=drop_last,
                  persistent_workers=pw,
                  prefetch_factor=4 if pw else None)
    if sampler is not None: kwargs["sampler"] = sampler
    else: kwargs["shuffle"] = shuffle
    return DataLoader(ds, **kwargs)


def ensure_split(root, val_frac=0.2, seed=SEED):
    if os.path.isdir(os.path.join(root, "train")): return root
    split_root = root.rstrip("/") + "_split"
    if os.path.isdir(os.path.join(split_root, "train")):
        print(f"  Using cached split at {split_root}"); return split_root
    print(f"  Creating split → {split_root} …")
    base   = datasets.ImageFolder(root)
    labels = np.array([y for _, y in base.samples])
    sss    = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    for split_name, indices in [("train", train_idx), ("val", val_idx)]:
        for idx in indices:
            src, cls_idx = base.samples[idx]
            dst_dir = os.path.join(split_root, split_name, base.classes[cls_idx])
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, os.path.basename(src))
            if not os.path.exists(dst):
                try: os.link(src, dst)
                except OSError: shutil.copy2(src, dst)
    return split_root


def _make_backbone(arch=None, img_size=None):
    arch     = arch     or CFG["arch"]
    img_size = img_size or CFG["img_size"]
    m = timm.create_model(arch, pretrained=False, num_classes=0,
                           img_size=img_size, dynamic_img_size=True)
    if hasattr(m, "set_attn_backend"):
        try: m.set_attn_backend("sdpa")
        except: pass
    return m


def load_backbone(backbone_path, device):
    ckpt  = torch.load(backbone_path, map_location="cpu")
    state = {k.replace("module.", "").replace("student_backbone.", ""): v
             for k, v in ckpt["model"].items() if "student_backbone" in k}
    backbone = _make_backbone()
    backbone.load_state_dict(state, strict=False)
    backbone.eval().to(device)
    for p in backbone.parameters(): p.requires_grad_(False)
    return backbone


def get_eval_transforms():
    val_tf = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225]),
    ])
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


@torch.inference_mode()
def extract_features(backbone, loader, device):
    all_feats, all_labels = [], []
    backbone.eval()
    use_amp = AMP_DTYPE is not None
    for x, y in loader:
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
            feats = backbone(x.to(device, non_blocking=True))
        all_feats.append(feats.float().cpu()); all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


# ── MultiSpectralDataset & BandAdapterBackbone ────────────────────────────────

class MultiSpectralDataset(Dataset):
    def __init__(self, root, n_channels=13):
        self.samples = []; self.n_channels = n_channels
        classes = sorted(d for d in os.listdir(root)
                         if os.path.isdir(os.path.join(root, d)))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            for fname in sorted(os.listdir(os.path.join(root, cls))):
                if fname.lower().endswith((".tif",".tiff",".npy",".png",".jpg")):
                    self.samples.append(
                        (os.path.join(root, cls, fname), self.class_to_idx[cls]))
    def __len__(self): return len(self.samples)
    def _normalise(self, x):
        mean = x.view(x.shape[0],-1).mean(1,keepdim=True).unsqueeze(-1)
        std  = x.view(x.shape[0],-1).std(1,keepdim=True).unsqueeze(-1).clamp(min=1e-6)
        return (x - mean) / std
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        ext = os.path.splitext(path)[1].lower()
        if ext in (".tif", ".tiff"):
            import rasterio
            with rasterio.open(path) as src:
                arr = src.read().astype(np.float32)
            n = self.n_channels
            arr = arr[:n] if arr.shape[0] >= n else np.concatenate(
                [arr, np.zeros((n-arr.shape[0], *arr.shape[1:]), dtype=np.float32)])
            x = torch.from_numpy(arr)
        elif ext == ".npy":
            arr = np.load(path).astype(np.float32)
            if arr.ndim==3 and arr.shape[2]==self.n_channels: arr=arr.transpose(2,0,1)
            x = torch.from_numpy(arr)[:self.n_channels]
        else:
            from PIL import Image
            img = Image.open(path).convert("RGB")
            x = transforms.Compose([transforms.Resize(256),
                transforms.CenterCrop(224), transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])(img)
            x = x.repeat(5,1,1)[:self.n_channels]; return x, label
        x = self._normalise(x)
        x = F.interpolate(x[None], size=224, mode="bilinear", align_corners=False)[0]
        return x, label


class BandAdapterBackbone(nn.Module):
    def __init__(self, backbone, in_channels=13, out_channels=3):
        super().__init__()
        self.adapter  = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.backbone = backbone
        nn.init.kaiming_normal_(self.adapter.weight)
    def forward(self, x): return self.backbone(self.adapter(x))


# ── Fixed eval_band_mismatch ──────────────────────────────────────────────────

def eval_band_mismatch_fixed(backbone_path, eurosat_rgb_dir,
                             eurosat_ms_dir=None, num_classes=10):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone  = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(backbone, val_ldr, device)
    feats  = F.normalize(feats.float(), dim=-1)
    sim_nd = feats @ feats.T
    sim_nd.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim_nd.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13)
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb   = BandAdapterBackbone(backbone, 13, 3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)
            tmp_head     = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt    = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()), lr=1e-3)

            for _ in range(5):
                adapter_bb.adapter.train(); tmp_head.train()
                for x, y in ms_ldr_train:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    # FIX: no_grad (not inference_mode) + .clone() so autograd
                    # can track the tmp_head forward / backward pass.
                    with torch.no_grad():
                        feat = backbone(adapter_bb.adapter(x)).clone()
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_feats, ms_lbl = [], []
            with torch.no_grad():
                for x, y in make_loader(ms_ds, 128, shuffle=False):
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True)).float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")
    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms is not None else None,
        "band_mismatch_delta": delta,
    }


# ── Load partial results & run §4.5 ──────────────────────────────────────────

BEST_CKPT      = os.path.join(CFG["output_dir"], f"dino_ep{CFG['epochs']}.pt")
eval_json_path = os.path.join(CFG["output_dir"], "dino_eval_results.json")

if os.path.exists(eval_json_path):
    with open(eval_json_path) as f:
        all_results = json.load(f)
    print("Loaded existing partial results from dino_eval_results.json")
else:
    # Paste your §4.1–4.4 log values here if the JSON wasn't saved
    all_results = {
        "model": "DINO", "checkpoint": BEST_CKPT,
        "classification": {}, "segmentation": {},
        "retrieval": {},     "representation": {},
    }
    print("No saved JSON — fill in the dict above from your logs, then re-run.")

print("\n" + "="*60)
print("§4.5  BAND MISMATCH ROBUSTNESS")
print("="*60)
all_results["band_mismatch"] = eval_band_mismatch_fixed(
    BEST_CKPT, CFG["eurosat_rgb_dir"], CFG["eurosat_ms_dir"])

out_path = os.path.join(CFG["output_dir"], "dino_eval_results.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nFinal results saved to {out_path}")
print(json.dumps(all_results, indent=2))

No saved JSON — fill in the dict above from your logs, then re-run.

§4.5  BAND MISMATCH ROBUSTNESS
  Recall@1 (RGB, 3-band) = 59.33%
  Recall@1 (MS, 13-band) = 40.90%
  Band mismatch penalty Δ = 18.43%

Final results saved to /kaggle/working/dino/dino_eval_results.json
{
  "model": "DINO",
  "checkpoint": "/kaggle/working/dino/dino_ep200.pt",
  "classification": {},
  "segmentation": {},
  "retrieval": {},
  "representation": {},
  "band_mismatch": {
    "recall1_rgb": 59.33,
    "recall1_ms": 40.9,
    "band_mismatch_delta": 18.43
  }
}
